# Workshop — Ingestion — **SOLUTION**

**Scenario:** You are a Data Engineer at RetailHub. Your task is to load raw customer and order data
from files into the Lakehouse, explore it, clean it, and write it to Delta tables — following the
Medallion Architecture pattern.

## Structure

| Task | Topic | Key Functions |
|------|-------|---------------|
| 01 | Setup & imports | `StructType`, functions |
| 02 | Load CSV — auto schema | `spark.read.format("csv")` |
| 03 | Load CSV — explicit schema | `StructType`, `StructField` |
| 04 | Explore data | `printSchema`, `describe`, `display` |
| 05 | Load JSON with schema | `spark.read.format("json")` |
| 06 | Select & filter | `select`, `filter`, `where` |
| 07 | Add and transform columns | `withColumn`, `cast`, `lit` |
| 08 | Conditional logic | `when` / `otherwise` |
| 09 | Aggregations | `groupBy`, `agg`, `count`, `sum`, `avg` |
| 10 | SQL — temporary views | `createOrReplaceTempView`, `spark.sql` |
| 11 | Write Bronze Delta table | `write.format("delta").saveAsTable` |
| 12 | BONUS — Join orders + customers | `join` |

In [0]:
%run ../setup/00_setup

In [0]:
CUSTOMERS_CSV  = f"{DATASET_PATH}/customers/customers.csv"
ORDERS_JSON    = f"{DATASET_PATH}/orders/orders_batch.json"
TARGET_TABLE   = f"{CATALOG}.{BRONZE_SCHEMA}.lab_ingestion_customers"

print(f"Customers : {CUSTOMERS_CSV}")
print(f"Orders    : {ORDERS_JSON}")
print(f"Target    : {TARGET_TABLE}")

---
## Task 01 — Imports

Import the following:
- `StructType`, `StructField` from `pyspark.sql.types`
- `StringType`, `IntegerType`, `DoubleType`, `TimestampType`, `DateType` from `pyspark.sql.types`
- `col`, `lit`, `current_timestamp`, `input_file_name`, `when`, `year`, `count`, `sum`, `avg`, `desc` from `pyspark.sql.functions`

In [0]:
from pyspark.sql.types import (
    StructType, StructField,
    StringType, IntegerType, DoubleType, TimestampType, DateType
)
from pyspark.sql.functions import (
    col, lit, current_timestamp, input_file_name, when,
    year, count, sum, avg, desc
)

In [0]:
# ── CHECK 01 ──────────────────────────────────────────────────────────
assert 'StructType'         in dir(), "Missing: StructType"
assert 'col'                in dir(), "Missing: col"
assert 'current_timestamp'  in dir(), "Missing: current_timestamp"
assert 'when'               in dir(), "Missing: when"
assert 'desc'               in dir(), "Missing: desc"
print("✓ Task 01 passed — imports OK")

---
## Task 02 — Load CSV with automatic schema inference

Load `CUSTOMERS_CSV` using `inferSchema=true` and `header=true`.
Store the result in `customers_infer_df`.
Then print the schema and display the first 5 rows.

In [0]:
customers_infer_df = (
    spark.read
        .format("csv")
        .option("header", "true")
        .option("inferSchema", "true")
        .load(CUSTOMERS_CSV)
)

customers_infer_df.printSchema()
display(customers_infer_df.limit(5))

In [0]:
# ── CHECK 02 ──────────────────────────────────────────────────────────
assert 'customers_infer_df' in dir(), "customers_infer_df not defined"
assert customers_infer_df.count() > 0, "DataFrame is empty"
assert 'customer_id' in customers_infer_df.columns, "Missing column: customer_id"
assert 'email'       in customers_infer_df.columns, "Missing column: email"
print(f"✓ Task 02 passed — {customers_infer_df.count():,} rows, {len(customers_infer_df.columns)} columns")

---
## Task 03 — Load CSV with explicit schema

Define a `StructType` for the customer dataset and use it to load the CSV.
Store the schema in `customers_schema` and the DataFrame in `customers_df`.

In [0]:
# Step 1 — define customers_schema
customers_schema = StructType([
    StructField("customer_id",       StringType(), nullable=True),  # primary key, not nullable
    StructField("first_name",        StringType(), nullable=True),
    StructField("last_name",         StringType(), nullable=True),
    StructField("email",             StringType(), nullable=True),
    StructField("phone",             StringType(), nullable=True),
    StructField("city",              StringType(), nullable=True),
    StructField("state",             StringType(), nullable=True),
    StructField("country",           StringType(), nullable=True),
    StructField("registration_date", StringType(), nullable=True),  # required field, not nullable
    StructField("customer_segment",  StringType(), nullable=True),
])

# Step 2 — load CUSTOMERS_CSV using the explicit schema
customers_df = (
    spark.read
        .format("csv")
        .option("header", "true")
        .schema(customers_schema)
        .option("enforceSchema", "true")  # Enforce schema nullability
        .load(CUSTOMERS_CSV)
)

customers_df.printSchema()
display(customers_df.limit(5))

In [0]:
# ── CHECK 03 ──────────────────────────────────────────────────────────
from pyspark.sql.types import DateType as _DT, StringType as _ST

assert 'customers_df'     in dir(), "customers_df not defined"
assert 'customers_schema' in dir(), "customers_schema not defined"
assert isinstance(customers_schema, StructType), "customers_schema must be StructType"

_schema_dict = {f.name: f.dataType for f in customers_df.schema.fields}
assert 'customer_id' in _schema_dict, "Missing column: customer_id"
assert 'registration_date'  in _schema_dict, "Missing column: registration_date"
assert isinstance(_schema_dict['registration_date'], _ST), \
    f"registration_date must be DateType, got {_schema_dict['registration_date']}"


assert customers_df.count() > 0, "DataFrame is empty"
print(f"✓ Task 03 passed — explicit schema OK, {customers_df.count():,} rows")

## Task 04 — Explore the customer DataFrame

Using `customers_df`, compute:
1. Total row count → store in `total_rows` (int)
2. Number of distinct countries → store in `distinct_countries` (int)
3. Summary statistics for all columns → store in `summary_df` (DataFrame from `.summary()`)

In [0]:
total_rows         = customers_df.count()
distinct_countries = customers_df.select("country").distinct().count()
summary_df         = customers_df.summary()

print(f"Total rows          : {total_rows:,}")
print(f"Distinct countries  : {distinct_countries}")
display(summary_df)

In [0]:
# ── CHECK 04 ──────────────────────────────────────────────────────────
assert isinstance(total_rows, int) and total_rows > 0, \
    "total_rows must be a positive integer"
assert isinstance(distinct_countries, int) and distinct_countries == 1, \
    "distinct_countries must be > 1"
assert hasattr(summary_df, 'columns'), "summary_df must be a DataFrame"
print(f"✓ Task 04 passed — {total_rows:,} rows, {distinct_countries} countries")

## Task 05 — Load JSON (Orders)

Load `ORDERS_JSON` using an explicit schema.
Store the schema in `orders_schema` and the DataFrame in `orders_df`.

In [0]:
# Step 1 — define orders_schema (8 fields)
orders_schema = StructType([
    StructField("order_id",        StringType(),  nullable=True),
    StructField("customer_id",     StringType(),  nullable=True),
    StructField("order_datetime",  StringType(),  nullable=True),
    StructField("quantity",        IntegerType(), nullable=True),
    StructField("unit_price",      DoubleType(),  nullable=True),
    StructField("total_amount",    DoubleType(),  nullable=True),
    StructField("payment_method",  StringType(),  nullable=True),
    StructField("status",          StringType(),  nullable=True),
])

# Step 2 — load ORDERS_JSON using the schema
orders_df = (
    spark.read
        .format("json")
        .schema(orders_schema)
        .load(ORDERS_JSON)
)

orders_df.printSchema()
display(orders_df.limit(5))

In [0]:
# ── CHECK 05 ──────────────────────────────────────────────────────────
assert 'orders_df'     in dir(), "orders_df not defined"
assert 'orders_schema' in dir(), "orders_schema not defined"

_ocols = orders_df.columns
for _c in ['order_id', 'customer_id', 'total_amount', 'payment_method']:
    assert _c in _ocols, f"Missing column: {_c}"

_odict = {f.name: f.dataType for f in orders_df.schema.fields}
assert isinstance(_odict.get('total_amount'), DoubleType), \
    "total_amount must be DoubleType"
assert isinstance(_odict.get('order_datetime'), StringType), \
    "order_datetime must remain StringType in Bronze/this task"

assert orders_df.count() > 0, "orders_df is empty"
print(f"✓ Task 05 passed — orders loaded: {orders_df.count():,} rows")

## Task 06 — Filter customers

From `customers_df` create two filtered DataFrames:

1. `customers_usa` — customers from the **USA** only
2. `customers_recent` — customers registered **in 2023 or later**

In [0]:
# Filter 1 — customers from USA only
customers_usa = customers_df.filter(col("country") == "USA")

# Filter 2 — customers registered in 2024 or later
customers_recent = customers_df.filter(year(col("registration_date")) >= 2024)

print(f"USA customers      : {customers_usa.count():,}")
print(f"Recent customers   : {customers_recent.count():,}")

In [0]:
# ── CHECK 06 ──────────────────────────────────────────────────────────
assert 'customers_usa'    in dir(), "customers_usa not defined"
assert 'customers_recent' in dir(), "customers_recent not defined"

_non_usa = customers_usa.filter(col("country") != "USA").count()
assert _non_usa == 0, f"customers_usa contains {_non_usa} non-USA rows"

_c_usa = customers_usa.count()
assert 0 < _c_usa == total_rows, "customers_usa must be a non-empty subset"

_old = customers_recent.filter(year(col("registration_date")) < 2023).count()
assert _old == 0, f"customers_recent contains {_old} rows from before 2023"

print(f"✓ Task 06 passed — USA: {_c_usa:,} | recent: {customers_recent.count():,}")

## Task 07 — Add and transform columns

From `customers_df` create `customers_enriched` with two additional columns:

1. `full_name` — concatenation of `first_name + " " + last_name`
2. `registration_year` — the year extracted from `registration_date` as IntegerType

In [0]:
from pyspark.sql.functions import concat_ws

customers_enriched = (
    customers_df
    .withColumn("full_name",
        concat_ws(" ", col("first_name"), col("last_name")))
    .withColumn("registration_year",
        year(col("registration_date")).cast(IntegerType()))
)

display(customers_enriched.select("customer_id", "full_name", "registration_date", "registration_year").limit(10))

In [0]:
# ── CHECK 07 ──────────────────────────────────────────────────────────
from pyspark.sql.types import IntegerType as _IntT

assert 'customers_enriched' in dir(), "customers_enriched not defined"
_ecols = customers_enriched.columns
assert 'full_name'          in _ecols, "Missing column: full_name"
assert 'registration_year'  in _ecols, "Missing column: registration_year"

_edict = {f.name: f.dataType for f in customers_enriched.schema.fields}
assert isinstance(_edict['registration_year'], _IntT), \
    f"registration_year must be IntegerType, got {_edict['registration_year']}"

_sample = customers_enriched.select("first_name", "last_name", "full_name").limit(50).collect()
for row in _sample:
    if row.first_name and row.last_name:
        expected = f"{row.first_name} {row.last_name}"
        assert row.full_name == expected, \
            f"full_name mismatch: expected '{expected}', got '{row.full_name}'"

print("✓ Task 07 passed — full_name and registration_year OK")

## Task 08 — Conditional column: customer tier

Add a `customer_tier` column to `customers_enriched`.
Store the result in `customers_tiered`.

| Condition | Tier |
|-----------|------|
| `registration_year` >= 2023 | `"New"` |
| `registration_year` >= 2021 | `"Regular"` |
| otherwise | `"Loyal"` |

In [0]:
customers_tiered = (
    customers_enriched
    .withColumn("customer_tier",
        when(col("registration_year") >= 2024, "New")
        .when(col("registration_year") >= 2021, "Regular")
        .otherwise("Loyal"))
)

display(customers_tiered.groupBy("customer_tier").count().orderBy("customer_tier"))

In [0]:
# ── CHECK 08 ──────────────────────────────────────────────────────────
assert 'customers_tiered' in dir(), "customers_tiered not defined"
assert 'customer_tier'    in customers_tiered.columns, \
    "Missing column: customer_tier"

_tiers = {r['customer_tier'] for r in customers_tiered.select("customer_tier").distinct().collect()}
assert _tiers <= {"New", "Regular", "Loyal"}, \
    f"Unexpected tier values: {_tiers}"
assert len(_tiers) >= 2, "Expected at least 2 distinct tiers in the dataset"

_null_tiers = customers_tiered.filter(col("customer_tier").isNull()).count()
assert _null_tiers == 0, f"customer_tier has {_null_tiers} null values — add .otherwise()"

print(f"✓ Task 08 passed — tiers: {sorted(_tiers)}")

## Task 09 — Aggregations

Using `orders_df`, compute aggregations grouped by `payment_method`.
Store the result in `payment_stats`, ordered by `total_revenue` descending.

In [0]:
payment_stats = (
    orders_df
    .groupBy("payment_method")
    .agg(
        count("*").alias("order_count"),
        sum("total_amount").alias("total_revenue"),
        avg("total_amount").alias("avg_order_value")
    )
    .orderBy(desc("total_revenue"))
)

display(payment_stats)

In [0]:
# ── CHECK 09 ──────────────────────────────────────────────────────────
assert 'payment_stats' in dir(), "payment_stats not defined"

_pcols = payment_stats.columns
for _c in ['payment_method', 'order_count', 'total_revenue', 'avg_order_value']:
    assert _c in _pcols, f"Missing column in payment_stats: {_c}"

_all_methods  = {r['payment_method'] for r in orders_df.select("payment_method").distinct().collect()}
_stat_methods = {r['payment_method'] for r in payment_stats.select("payment_method").collect()}
assert _all_methods == _stat_methods, "payment_stats does not cover all payment methods"

_totals = [r['total_revenue'] for r in payment_stats.collect()]
assert _totals == sorted(_totals, reverse=True), "Result must be ordered by total_revenue DESC"

print(f"✓ Task 09 passed — {len(_stat_methods)} payment methods aggregated")

## Task 10 — Temporary views and SQL

1. Register `customers_df` as a temp view named `"customers_view"`
2. Register `orders_df` as a temp view named `"orders_view"`
3. Query the top 5 countries by number of customers. Store the result in `top_countries`.

In [0]:
# Step 1 — register both DataFrames as temp views
customers_df.createOrReplaceTempView("customers_view")
orders_df.createOrReplaceTempView("orders_view")

# Step 2 — query top 5 countries by customer count
top_city = spark.sql("""
    SELECT
        city,
        COUNT(*) AS customer_count
    FROM customers_view
    GROUP BY city
    ORDER BY customer_count DESC
    LIMIT 5
""")

display(top_city)

In [0]:
# ── CHECK 10 ──────────────────────────────────────────────────────────
assert 'top_city' in dir(), "top_countries not defined"

_views = {r.viewName for r in spark.sql("SHOW VIEWS").collect()}
assert 'customers_view' in _views, "Temp view 'customers_view' not found"
assert 'orders_view'    in _views, "Temp view 'orders_view' not found"

assert 'city'        in top_city.columns, "Missing column: country"
assert 'customer_count' in top_city.columns, "Missing column: customer_count"
assert top_city.count() == 5, f"Expected 5 rows, got {top_city.count()}"

_counts = [r['customer_count'] for r in top_city.collect()]
assert _counts == sorted(_counts, reverse=True), "Result must be ordered by customer_count DESC"

print(f"✓ Task 10 passed — views registered, top countries: {[r['city'] for r in top_city.collect()]}")

## Task 11 — Write Bronze Delta table

Write `customers_tiered` to a managed Delta table at `TARGET_TABLE`.
Add `_load_ts` and `_source_file` metadata columns first.
Store the final DataFrame in `customers_bronze`.

In [0]:
# Step 1 — add Bronze metadata columns
customers_bronze = (
    customers_tiered
    .withColumn("_load_ts",     current_timestamp())
    .withColumn("_source_file", lit("customers.csv"))
)

# Step 2 — write to managed Delta table
(
    customers_bronze.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(TARGET_TABLE)
)

print(f"Written to: {TARGET_TABLE}")
display(spark.table(TARGET_TABLE).limit(5))

In [0]:
# ── CHECK 11 ──────────────────────────────────────────────────────────
from pyspark.sql.types import TimestampType as _TsT

assert 'customers_bronze' in dir(), "customers_bronze not defined"

_tbl   = spark.table(TARGET_TABLE)
_tcols = _tbl.columns
assert '_load_ts'      in _tcols, "Missing column: _load_ts"
assert '_source_file'  in _tcols, "Missing column: _source_file"
assert 'customer_tier' in _tcols, "Missing column: customer_tier (from Task 08)"

_tdict = {f.name: f.dataType for f in _tbl.schema.fields}
assert isinstance(_tdict['_load_ts'], _TsT), "_load_ts must be TimestampType"

_row_count = _tbl.count()
assert _row_count > 0, "Delta table is empty"

_src_vals = {r['_source_file'] for r in _tbl.select("_source_file").distinct().collect()}
assert 'customers.csv' in _src_vals, "_source_file must contain 'customers.csv'"

print(f"✓ Task 11 passed — {_row_count:,} rows written to {TARGET_TABLE}")

## Task 12 BONUS — Join orders + customers

Create `orders_enriched` by joining `orders_df` (left) with `customers_df` (right)
on `customer_id`. Use a **left join**.

Select only: `order_id`, `customer_id`, `first_name`, `last_name`, `country`, `total_amount`, `payment_method`.

Then compute `orders_by_country`: count of orders and total revenue per country, ordered by `total_revenue` DESC.

In [0]:
# Step 1 — left join orders_df with customers_df on customer_id
# Use df["col"] notation to resolve the ambiguous customer_id column
orders_enriched = (
    orders_df
    .join(customers_df,
          orders_df["customer_id"] == customers_df["customer_id"],
          "left")
    .select(
        orders_df["order_id"],
        orders_df["customer_id"],
        customers_df["first_name"],
        customers_df["last_name"],
        customers_df["country"],
        orders_df["total_amount"],
        orders_df["payment_method"]
    )
)

# Step 2 — group by country: compute order_count and total_revenue (DESC)
orders_by_country = (
    orders_enriched
    .groupBy("country")
    .agg(
        count("*").alias("order_count"),
        sum("total_amount").alias("total_revenue")
    )
    .orderBy(desc("total_revenue"))
)

display(orders_by_country.limit(10))

In [0]:
# ── CHECK 12 ──────────────────────────────────────────────────────────
assert 'orders_enriched'   in dir(), "orders_enriched not defined"
assert 'orders_by_country' in dir(), "orders_by_country not defined"

_ecols = orders_enriched.columns
for _c in ['order_id', 'customer_id', 'first_name', 'country', 'total_amount']:
    assert _c in _ecols, f"Missing column in orders_enriched: {_c}"

assert orders_enriched.count() == orders_df.count(), \
    "Left join must preserve all orders (same row count as orders_df)"

assert 'total_revenue' in orders_by_country.columns or \
       'total_amount'  in orders_by_country.columns, \
    "orders_by_country must have a revenue column"

print(f"✓ Task 12 BONUS passed — {orders_enriched.count():,} enriched orders, "
      f"{orders_by_country.count()} countries")


## Workshop Complete!

| Task | Topic | Status |
|------|-------|--------|
| 01 | Imports | ✓ |
| 02 | CSV — inferSchema | ✓ |
| 03 | CSV — explicit schema | ✓ |
| 04 | Exploration | ✓ |
| 05 | JSON with schema | ✓ |
| 06 | Filter | ✓ |
| 07 | withColumn, cast | ✓ |
| 08 | when / otherwise | ✓ |
| 09 | groupBy + agg | ✓ |
| 10 | Temp views + SQL | ✓ |
| 11 | Write Bronze Delta | ✓ |
| 12 | Join + aggregation | ✓ |

**Next step:** `WORKSHOP_delta.ipynb` — ACID, Time Travel, MERGE, VACUUM